[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/polars-certified/notebooks/day-06-joins-concatenation-and-reshaping.ipynb#scrollTo=a1f2e3b4)

---
# Day 6 · Joins, Concatenation and Reshaping
**certified-journeys / polars-certified** &nbsp;|&nbsp; Intermediate

> **Goal for today:** Master all Polars join types, combining DataFrames, and wide↔long transformations.

---
## Why joins matter in Polars

Polars implements joins as a **hash join** by default — extremely fast and memory-efficient.
Understanding which join type to use avoids silent data loss or unexpected row explosions.

| Join type | Rows kept | Use when |
|---|---|---|
| `inner` | Match in both | You only want confirmed matches |
| `left` | All left + matches from right | Left is the primary table |
| `right` | All right + matches from left | Right is the primary table |
| `full` | All rows from both | You can't afford to lose any row |
| `cross` | Every left row × every right row | Cartesian product (careful: O(n×m)) |
| `semi` | Left rows that DO match right | Existence check |
| `anti` | Left rows that DON'T match right | Find missing / unmatched rows |

In [ ]:
%pip install -q polars

---
## Step 1 · Create synthetic datasets

We need two related tables: **customers** and **orders**.
Notice that customer_id 4 has no orders, and order 105 references a non-existent customer —
these edge cases let us see the difference between join types clearly.

In [ ]:
import polars as pl

# Customers table: customer_id 4 exists but has no orders
customers = pl.DataFrame({
    "customer_id": [1, 2, 3, 4],
    "name":        ["Alice", "Bob", "Carol", "Dave"],
    "country":     ["US", "UK", "DE", "FR"],
    "credit_limit": [500.0, 300.0, 800.0, 200.0],
})

# Orders table: order 105 references customer_id 99 which does NOT exist
orders = pl.DataFrame({
    "order_id":    [101, 102, 103, 104, 105],
    "customer_id": [1, 2, 1, 3, 99],          # 99 is an orphan
    "amount":      [450.0, 120.0, 600.0, 350.0, 50.0],
    "date":        ["2024-01-10", "2024-01-15", "2024-02-01",
                    "2024-02-20", "2024-03-05"],
})

print("customers:", customers.shape)
print(customers)
print("\norders:", orders.shape)
print(orders)

**What just happened?**
- We created two DataFrames with a deliberate mismatch: customer 4 (Dave) has no orders, and order 105 has an orphan customer_id 99.
- **These mismatches are the key test cases** — watch how each join type handles them differently.
- `credit_limit` is included for the `join_where` example in Step 3.
- Polars infers dtypes automatically: `Int64` for IDs, `Float64` for amounts, `String` for dates.

---
## Step 2 · All six join types

We run the same join with 6 different `how=` values and print the row count each time.
Watch how the count changes — that tells you what each join includes or drops.

In [ ]:
join_types = ["inner", "left", "right", "full", "semi", "anti"]

for how in join_types:
    result = customers.join(orders, on="customer_id", how=how)
    print(f"{how:8s}  rows={result.shape[0]:2d}  cols={result.shape[1]}")

# Show the full inner join to inspect columns
print("\n--- inner join result ---")
print(customers.join(orders, on="customer_id", how="inner"))

# Full (outer) join: Dave (id=4) and orphan order 105 both appear
print("\n--- full join result ---")
print(customers.join(orders, on="customer_id", how="full"))

**What just happened?**
- **inner (4 rows):** Only customers who have matching orders — Dave (id=4) and orphan order 105 are both dropped.
- **left (5 rows):** All 4 customers kept; Dave appears with nulls. Orphan order 105 is still dropped.
- **full (6 rows):** Every row from both tables — Dave and orphan order 105 both appear with nulls.
- **semi (3 rows):** Returns only the customer columns for customers who DO have orders — useful for existence checks.
- **anti (1 row):** Returns only customer columns for customers with NO orders — Dave is the only result.

---
## Step 3 · join_where for non-equi (inequality) joins

`join_where` lets you join on arbitrary boolean conditions, not just equality.
Classic use case: find orders where the amount **exceeds** the customer's credit limit.
This requires a cross-product base — Polars handles this efficiently.

In [ ]:
# join_where: find each order that EXCEEDS the matching customer's credit limit
# Step 1: standard equi-join to attach credit_limit to each order
orders_with_limit = orders.join(
    customers.select(["customer_id", "credit_limit"]),
    on="customer_id",
    how="left",
)
print("Orders with credit limits:")
print(orders_with_limit)

# Step 2: filter for orders exceeding the limit
over_limit = orders_with_limit.filter(pl.col("amount") > pl.col("credit_limit"))
print("\nOrders OVER credit limit:")
print(over_limit)

# join_where native syntax (Polars >= 0.20): cross + condition in one call
# This is the idiomatic approach for non-equi joins in Polars
result_jw = customers.join_where(
    orders,
    pl.col("customer_id") == pl.col("customer_id_right"),  # equi part
    pl.col("amount") > pl.col("credit_limit"),              # inequality part
)
print("\njoin_where result (amount > credit_limit):")
print(result_jw)

**What just happened?**
- `join_where` combines an equality condition with an inequality condition in a single join operation.
- **Order 103 (Alice, $600)** exceeds her $500 credit limit — the only flagged order.
- The `_right` suffix is Polars' way of disambiguating columns that exist in both DataFrames.
- **Always prefer a standard equi-join + filter** when possible; use `join_where` only when the inequality must be evaluated during join (e.g., range joins, temporal validity windows).

---
## Step 4 · Concatenation: vertical, diagonal, and horizontal

Three ways to combine DataFrames:

| Method | Direction | Schema requirement |
|---|---|---|
| `pl.concat(how="vertical")` | Stack rows | Schemas must match exactly |
| `pl.concat(how="diagonal")` | Stack rows | Schemas can differ (nulls fill gaps) |
| `df.hstack(other)` | Stack columns | Row counts must match |

In [ ]:
# Vertical concat: same schema, just append rows
new_customers = pl.DataFrame({
    "customer_id":  [5, 6],
    "name":         ["Eve", "Frank"],
    "country":      ["JP", "BR"],
    "credit_limit": [1000.0, 750.0],
})

all_customers = pl.concat([customers, new_customers], how="vertical")
print(f"vertical concat: {all_customers.shape[0]} rows")
print(all_customers)

# Diagonal concat: mismatched schemas — missing columns become null
customers_no_limit = customers.drop("credit_limit")  # different schema
diagonal_result = pl.concat([customers_no_limit, new_customers], how="diagonal")
print(f"\ndiagonal concat (mismatched schemas): {diagonal_result.shape}")
print(diagonal_result)

# Horizontal stack: glue columns side by side (row counts must match)
scores = pl.DataFrame({"score": [88, 72, 95, 61]})
customers_with_score = customers.hstack(scores)  # both have 4 rows
print("\nhstack result:")
print(customers_with_score)

# What happens with a length mismatch?
try:
    customers.hstack(pl.DataFrame({"score": [1, 2]}))  # only 2 rows!
except Exception as e:
    print(f"\nhstack error (expected): {e}")

**What just happened?**
- **`how="vertical"`** requires identical schemas — it's the fastest concat, a simple memory append.
- **`how="diagonal"`** fills missing columns with `null` — essential when merging data from different sources with evolving schemas.
- **`hstack`** glues columns together without a join key — only valid when row order is guaranteed to align.
- **Length mismatch in `hstack` raises a `ShapeError`** immediately — Polars is strict about this, which protects you from silent misalignment bugs.

---
## Step 5 · Pivot: long → wide

`pivot()` takes a column of values and spreads them as new column headers.
Classic use case: monthly sales per product as separate columns for a spreadsheet-friendly view.

In [ ]:
# Long-format sales data: product, month, revenue
sales_long = pl.DataFrame({
    "product": ["A", "A", "A", "B", "B", "B", "C", "C", "C"],
    "month":   ["Jan", "Feb", "Mar", "Jan", "Feb", "Mar", "Jan", "Feb", "Mar"],
    "revenue": [100, 120, 90, 200, 180, 220, 50, 60, 55],
})

print("Long format:")
print(sales_long)

# pivot: each unique 'month' value becomes its own column
sales_wide = sales_long.pivot(
    values="revenue",     # values to fill the new columns
    index="product",      # rows — what identifies each row
    on="month",           # column whose values become new column names
    aggregate_function="sum",  # in case of duplicates
)
print("\nWide format (pivoted):")
print(sales_wide)

**What just happened?**
- `pivot()` reshapes 9 rows × 3 cols into 3 rows × 4 cols — one row per product, one column per month.
- The `on=` parameter names the column whose **distinct values** become new column headers.
- `aggregate_function="sum"` handles duplicates gracefully — always specify this to avoid ambiguity errors.
- **Wide format** is great for presentation and certain ML features; **long format** is better for Polars operations like groupby and window functions.

---
## Step 6 · Unpivot: wide → long

`unpivot()` (formerly `melt()`) is the inverse of pivot — it collapses multiple columns
into a single key-value pair structure. Essential for tidy data workflows.

In [ ]:
# Start from the wide format we just created
print("Wide format:")
print(sales_wide)

# unpivot: collapse the month columns back into a single 'month' column
sales_long_again = sales_wide.unpivot(
    index="product",                    # columns to keep as-is (the row identifier)
    on=["Jan", "Feb", "Mar"],           # columns to collapse into key-value rows
    variable_name="month",              # name for the new key column
    value_name="revenue",              # name for the new value column
)
print("\nLong format (after unpivot):")
print(sales_long_again.sort(["product", "month"]))

**What just happened?**
- `unpivot()` collapsed 3 month columns back into 9 rows — reversing the pivot exactly.
- `index=` specifies which columns stay as row identifiers (not melted).
- `on=` lists which columns to collapse — omitting this would melt **all** non-index columns.
- **`unpivot` is the Polars 0.20+ name**; older code uses `melt()` which still works but is deprecated.

---
## Challenge

You have a wide sales table (product × month) and a product metadata table.
Your task:
1. Unpivot the wide table to long format
2. Join with the metadata table to add the `category` column
3. Aggregate total revenue per category per month

In [ ]:
# Wide sales table (given)
wide_sales = pl.DataFrame({
    "product": ["A", "B", "C", "D"],
    "Jan": [100, 200, 50, 80],
    "Feb": [120, 180, 60, 90],
    "Mar": [90, 220, 55, 110],
})

# Product metadata table (given)
product_meta = pl.DataFrame({
    "product":  ["A", "B", "C", "D"],
    "category": ["Electronics", "Electronics", "Books", "Books"],
})

# TODO: Step 1 — unpivot wide_sales so you have columns: product, month, revenue
long_sales = ...  # your code here

# TODO: Step 2 — join long_sales with product_meta on 'product'
enriched = ...  # your code here

# TODO: Step 3 — group by category + month, sum revenue
summary = ...  # your code here

# print(summary)

---
## Day 6 recap

| Concept | Key method | When to use |
|---|---|---|
| Equi joins | `df.join(other, on=, how=)` | Most joins — 7 types available |
| Inequality joins | `df.join_where(other, cond1, cond2)` | Range joins, validity windows |
| Vertical stack | `pl.concat([a,b], how="vertical")` | Same schema, append rows |
| Diagonal stack | `pl.concat([a,b], how="diagonal")` | Different schemas, fill nulls |
| Horizontal stack | `df.hstack(other)` | Align columns by row position |
| Long → wide | `df.pivot(values=, index=, on=)` | Reporting, ML features |
| Wide → long | `df.unpivot(index=, on=)` | Tidy data, groupby operations |

> **Tip:** Anti-join (`how="anti"`) is one of the most underused features in Polars. It returns rows in A that have **no match** in B — perfect for finding orphaned records, gaps in coverage, or data quality checks. `customers.join(orders, on="customer_id", how="anti")` returns every customer who has never placed an order.

---
**What's next → Day 7:** I/O — Parquet, CSV, JSON, and connecting Polars to DuckDB for SQL-on-DataFrame.

Mark Day 6 complete in your [tracker](../index.html).